In [2]:
from datasets import load_from_disk

ds = load_from_disk("../musiccaps/dataset_audio")
sample = ds[0]

Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

In [3]:
from transformers import ClapProcessor, ClapModel
import torch
import librosa
import numpy as np

processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").eval()

# Get audio from dataset
audio_data = ds[0]["audio"]["array"]
original_sr = ds[0]["audio"]["sampling_rate"]  # 44100

# Resample to 48000 if needed
if original_sr != 48000:
    audio_data = librosa.resample(
        audio_data, 
        orig_sr=original_sr, 
        target_sr=48000
    )

# Process with CLAP
inputs = processor(
    audios=audio_data,
    sampling_rate=48000,
    return_tensors="pt"
)

with torch.no_grad():
    audio_emb = clap.get_audio_features(**inputs)

print(audio_emb.shape)

/home/aliozkaya/miniconda3/envs/musicgen/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


torch.Size([1, 512])
